# TokaMark `task_1-3` Sensor-Robustness Study (Gupta arXiv 2607.11915) on Kaggle

**Goal:** Run the complete sensor-failure robustness evaluation on a *trained, frozen* task_1-3 model across 6 failure scenarios (random dropout, channel ablation, category importance, temporal gap, correlated group failure, disruption-proximate) and 3 imputation modes (zero, train-mean, forward-fill).

**Requirements on Kaggle:**
- **Accelerator:** GPU T4 (x1 or x2).
- **Internet:** ON (for downloading test shots from Hugging Face).
- **Input Data Attached:** Add the output of your training notebook (`Fusion_recon` or `run_40gb`) to `/kaggle/input/`.
- **Execution:** Click **"Save Version" -> "Save & Run All (Commit)"** to run headlessly in the background (~25-35 minutes total).


In [ ]:
import os, sys, time, shutil, subprocess, json, glob

# ================= CONFIG =================
DATA_SPLIT    = "random"  # "random" or "temporal"
MODEL_TYPE    = "mlp"     # "mlp" (CNN+MLP champion) or "lstm" (CNN+LSTM benchmark)
BATCH         = 512       # batch size for test inference
SEED          = 42
DL_WORKERS    = 0         # 0 = main process (hang-proof on Kaggle)
ROB_SCENARIOS = "all"     # or comma-separated list of scenarios
ROB_IMPUTE    = "zero,mean,ffill"
# ==========================================

T0 = time.time()
os.environ["TM_T0"] = str(T0)

def run(cmd, check=True):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ)
    for line in p.stdout: print(line, end="", flush=True)
    p.wait()
    if check and p.returncode: raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

print("=== System Resources ===")
run("nvidia-smi -L; nproc; free -g | head -2; df -h /kaggle/working /tmp 2>/dev/null", check=False)
run("mount -o remount,size=8G /dev/shm 2>/dev/null || sudo mount -o remount,size=8G /dev/shm", check=False)
run("df -h /dev/shm", check=False)

W = "/kaggle/working"
cands = [d for d in ("/kaggle/temp", "/tmp", "/kaggle/working") if os.path.isdir(d)]
DATA_ROOT = max(cands, key=lambda d: shutil.disk_usage(d).free) + "/tokamark_data"
print(f"data root: {DATA_ROOT} | free GB: {shutil.disk_usage(os.path.dirname(DATA_ROOT)).free/1e9:.2f}")

# Install necessary runtime dependencies
run("python -m pip install -q zarr==3.1.5 xarray 'fsspec==2025.3.0' 's3fs==2025.3.0' "
    "huggingface_hub pyyaml scipy requests filelock torchinfo==1.8.0", check=True)
run("python -c \"import zarr, xarray, s3fs, huggingface_hub, torchinfo, requests, filelock; print('dependency imports OK')\"", check=True)

# ==============================================================================
# RESTORE TRAINED ARTIFACTS FROM /kaggle/input (Handles DATASETS and NOTEBOOKS)
# ==============================================================================
print("\n=== Checking /kaggle/input for trained model artifacts ===")
input_dirs = glob.glob("/kaggle/input/*")
print("Input folders detected:", [os.path.basename(d) for d in input_dirs])

restored = False

# Priority 1: Direct match in dataset or notebook folders (e.g. /kaggle/input/results/out)
for cand in input_dirs:
    cand_out = os.path.join(cand, "out")
    if os.path.isdir(cand_out):
        print(f"Found artifacts in input directory: {cand}")
        run(f"mkdir -p '{W}/out' && cp -r '{cand_out}/'* '{W}/out/'")
        for r in ("tokamark", "tokamark_baseline"):
            r_src = os.path.join(cand, r)
            if os.path.isdir(r_src) and not os.path.isdir(f"{W}/{r}"):
                print(f"Restoring repo {r} from {cand}...")
                run(f"cp -r '{r_src}' '{W}/'")
        restored = True
        break

# Priority 2: Check for any results.zip archive in /kaggle/input
if not restored:
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".zip") and any(k in f.lower() for k in ("result", "fusion", "run", "40gb")):
                fpath = os.path.join(root, f)
                print(f"Extracting zip archive: {fpath} -> {W}/")
                run(f"unzip -q -o '{fpath}' -d {W}/")
                restored = True
                break
        if restored:
            break

# Priority 3: Deep recursive search for best_model.pt
if not restored:
    for root, dirs, files in os.walk("/kaggle/input"):
        if "best_model.pt" in files:
            cur = root
            while cur and cur != "/kaggle/input":
                if os.path.basename(cur) == "out":
                    print(f"Found out directory at {cur}, copying to {W}/out...")
                    run(f"mkdir -p '{W}/out' && cp -r '{cur}/'* '{W}/out/'")
                    restored = True
                    break
                cur = os.path.dirname(cur)
            if restored:
                break

# Verify and position the restored model checkpoint
expected_pt = f"{W}/out/{DATA_SPLIT}/model/best_model.pt"
if not os.path.exists(expected_pt):
    for root, dirs, files in os.walk(W):
        if "best_model.pt" in files:
            src_pt = os.path.join(root, "best_model.pt")
            run(f"mkdir -p '{W}/out/{DATA_SPLIT}/model' && cp '{src_pt}' '{expected_pt}'")
            print(f"[OK] Located and positioned model checkpoint to: {expected_pt}")
            break

if not os.path.exists(expected_pt):
    raise RuntimeError("Missing best_model.pt! Please attach your trained model results archive to /kaggle/input.")

print(f"[OK] Model successfully verified at: {expected_pt}")
expected_sel = f"{W}/out/{DATA_SPLIT}/selection.json"
if os.path.exists(expected_sel):
    print(f"[OK] Selection file verified at: {expected_sel}")
else:
    print(f"[INFO] selection.json not in archive - Step 1 (prep) will download official {DATA_SPLIT} test split automatically.")

# Ensure repos exist (clone if not copied from input)
REPOS = (("tokamark", "1a200f4"), ("tokamark_baseline", "1e7c489"))
for name, sha in REPOS:
    if not os.path.isdir(f"{W}/{name}"):
        run(f"git clone -q https://github.com/UKAEA-IBM-STFC-Fusion-FMs/{name}.git {W}/{name}")
    run(f"cd {W}/{name} && git checkout -q {sha} && git log -1 --format='{name} @ %h'")

env = dict(
    TM_SEED=SEED,
    TM_DATA_ROOT=DATA_ROOT,
    TM_WORK=W,
    TM_TOKAMARK=f"{W}/tokamark",
    TM_BASELINE=f"{W}/tokamark_baseline",
    TM_DL_WORKERS=DL_WORKERS,
    TM_GIT_REVS=",".join(f"{n}@{s}" for n, s in REPOS),
    TM_BATCH=BATCH,
    TM_MODEL=MODEL_TYPE,
    TM_ROB_SCENARIOS=ROB_SCENARIOS,
    TM_ROB_IMPUTE=ROB_IMPUTE,
    TM_ROB_SEED=SEED,
    TM_ROB_BATCH=BATCH,
)
os.environ.update({k: str(v) for k, v in env.items()})
print("Environment configured successfully.")


In [ ]:
%%writefile /kaggle/working/pipeline.py
#!/usr/bin/env python
"""TokaMark task_1-3 CNN baseline on Kaggle: prep | train | eval | plot | summary.

Usage: python pipeline.py {prep,train,eval,plot,summary} [--split random|temporal]
All settings come from environment variables (see CFG below), set by the notebook.

Protocol notes (deviations from the official run_training.py / run_evaluation.py):
- Custom step-based training loop mirroring src/trainer.py:BatchStepTrainer (same
  Adam(lr, wd=1e-4) + MultiOutputMSELoss, validate_every, patience, drop_last=True),
  extended with: wall-clock deadline, periodic + atomic checkpointing, resume from
  last.pt, CUDA-OOM reporting, and non-finite-loss skipping.
- Shots are a random subset OF THE OFFICIAL SPLIT LISTS (membership never mixed),
  chosen under a byte budget; official stats/outlier/split files always match the
  split protocol. test uses shuffle_windows=False (official eval behaviour).
"""
import os, sys, json, time, math, random, shutil, zipfile, argparse, subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

T0 = float(os.environ.get("TM_T0", time.time()))  # session start (set by notebook)


def env(k, d):
    return type(d)(os.environ.get(k, d))


CFG = dict(
    task=env("TM_TASK", "task_1-3"),
    model=env("TM_MODEL", "mlp"),                         # mlp (CNN+MLP) or lstm (CNN+LSTM)
    seed=env("TM_SEED", 23),
    repo_id="UKAEA-IBM-STFC/tokamark-v1",
    work=env("TM_WORK", "/kaggle/working"),
    data_root=env("TM_DATA_ROOT", "/tmp/tokamark_data"),
    tokamark_dir=env("TM_TOKAMARK", "/kaggle/working/tokamark"),
    baseline_dir=env("TM_BASELINE", "/kaggle/working/tokamark_baseline"),
    data_budget_gb=env("TM_DATA_GB", 40.0),
    prep_deadline_h=env("TM_PREP_DEADLINE_H", 1.75),      # hours since T0: stop downloading
    train_deadline_h=env("TM_TRAIN_DEADLINE_H", 8.0),     # hours since T0: stop training
    batch_size=env("TM_BATCH", 512),
    validate_every=env("TM_VALIDATE_EVERY", 100),
    patience=env("TM_PATIENCE", 10),
    lr=1e-4,
    weight_decay=env("TM_WEIGHT_DECAY", 1e-4),            # official Adam weight decay
    train_from_scratch=env("TM_TRAIN_FROM_SCRATCH", "0") == "1",
    max_steps=env("TM_MAX_STEPS", 10_000_000),
    dl_workers=env("TM_DL_WORKERS", -1),                  # -1 -> auto; 0 -> main process only; N -> N workers
    use_dp=env("TM_USE_DP", "auto"),                      # auto|0|1
    ckpt_every_s=env("TM_CKPT_EVERY_S", 600),
    force_reselect=env("TM_FORCE_RESELECT", "0") == "1",
    plot_windows=env("TM_PLOT_WINDOWS", 3),
    git_revs=env("TM_GIT_REVS", "unknown"),
    index_threads=env("TM_INDEX_THREADS", 12),            # concurrency; global req/s throttle (_RL_RPS) is the real limiter
    index_stop_gb=env("TM_INDEX_STOP_GB", 0.0),           # 0 -> auto: max(3, 1.6 x data budget)
    index_min_tars=env("TM_INDEX_MIN_TARS", 6),           # spread subset across >= this many archives
)


def log(*a):
    print(f"[{(time.time() - T0) / 60:7.1f}m]", *a, flush=True)


def paths(split):
    out = os.path.join(CFG["work"], "out", split)
    return dict(
        out=out,
        data=os.path.join(CFG["data_root"], split),
        index=os.path.join(CFG["work"], "out", "tar_index.json"),
        sel=os.path.join(out, "selection.json"),
        model_dir=os.path.join(out, "model") + "/",
        results=os.path.join(out, "results") + "/",
        figures=os.path.join(out, "figures") + "/",
    )


def setup_imports():
    for p in (os.path.join(CFG["tokamark_dir"], "src"), CFG["baseline_dir"]):
        if p not in sys.path:
            sys.path.insert(0, p)
    os.chdir(CFG["baseline_dir"])


def split_files(split):
    """Official split CSV + stats + outlier file; ALWAYS matched per protocol."""
    setup_imports()
    from tokamark.tools.path import (RANDOM_SPLIT_TOKAMARK_DATA_SPLITS_FILE as RS, RANDOM_SPLIT_SIGNALS_STATS_FILE as RT,
                                     TEMPORAL_SPLIT_TOKAMARK_DATA_SPLITS_FILE as TS, TEMPORAL_SPLIT_SIGNALS_STATS_FILE as TT)
    from MAST_tools.utils.path_utils import RANDOM_SPLIT_OUTLIER_METADATA_FILE as RO, TEMPORAL_SPLIT_OUTLIER_METADATA_FILE as TO
    if split == "random":
        return RS, RT, RO
    if split == "temporal":
        return TS, TT, TO
    raise ValueError(f"unknown split {split}")


def patch_stats_module(stats):
    """Point get_signals_metadata at the stats file of the CURRENT split protocol.

    Upstream get_task_metadata() always reads the RANDOM-split stats file, and
    tokamark.evaluator binds get_signals_metadata at import time; patch both module
    references so metadata and NRMSE normalisation always match the protocol.
    """
    setup_imports()
    import tokamark.tasks as _tt
    import tokamark.evaluator as _te
    if getattr(_tt, "_tm_patched", None) == stats:
        return
    orig = getattr(_tt, "_tm_orig", None) or _tt.get_signals_metadata
    _tt._tm_orig = orig
    patched = lambda file_path=stats, _o=orig: _o(file_path=file_path)  # noqa: E731
    _tt.get_signals_metadata = patched
    _te.get_signals_metadata = patched
    _tt._tm_patched = stats
    log(f"patched get_signals_metadata -> {os.path.basename(stats)}")


def first_batch(loader, tries=100):
    """Return the first non-empty batch (collate_fn may yield None for empty batches)."""
    it = iter(loader)
    for _ in range(tries):
        b = next(it, None)
        if b is not None:
            return b
    raise RuntimeError("loader produced no non-empty batch")


def first_batch_timed(loader, tag="first batch", tries=100, timeout_s=1800):
    """first_batch with a heartbeat + hard wall-clock cap.

    A silent hang here (workers stuck on IPC / fork-after-CUDA / slow zarr reads) used
    to look like training forever; now it is visible every 30 s and fails loudly.
    """
    import threading
    res, done = {}, threading.Event()

    def _work():
        try:
            res["b"] = first_batch(loader, tries)
        except BaseException as e:  # noqa
            res["e"] = e
        finally:
            done.set()

    threading.Thread(target=_work, daemon=True).start()
    t0 = time.time()
    while not done.wait(30):
        log(f"  waiting for {tag}... {time.time() - t0:.0f}s (workers are still producing it)")
        if time.time() - t0 > timeout_s:
            raise RuntimeError(
                f"{tag} did not arrive within {timeout_s}s. This is a dataloader hang, not training. "
                f"Try TM_DL_WORKERS=0 (main-process loading, no workers/IPC) or a smaller TM_BATCH "
                f"({CFG['batch_size']} -> 128/64).")
    if "e" in res:
        raise res["e"]
    return res["b"]


# ======================================================================================
# PREP: index tars with range requests, select shots per official split, fetch members
# ======================================================================================
def hf_headers():
    tok = os.environ.get("HF_TOKEN")
    return {"Authorization": f"Bearer {tok}"} if tok else {}


import threading
_TLS = threading.local()


def _session():
    """Per-thread requests.Session: TLS keep-alive turns ~1-3 s/request into ~0.2 s."""
    s = getattr(_TLS, "session", None)
    if s is None:
        import requests
        s = _TLS.session = requests.Session()
    return s


class _Deadline(Exception):
    pass


_RL_LOCK = threading.Lock()
_RL_NEXT = [0.0]        # earliest wall-clock time the next request may start (shared throttle)
_RL_RPS = [float(os.environ.get("TM_HF_RPS", 8))]    # current allowed req/s (adaptive)
_RL_RPS_MIN = max(1.0, float(os.environ.get("TM_HF_RPS_MIN", 2)))    # floor when throttled
_RL_429 = [0]           # consecutive-ish 429 counter (decays on success)


def _throttle():
    """Block until this thread may issue the next HF range request.

    All threads share one token budget (req/s). The rate is ADAPTIVE: each 429
    halves it (down to _RL_RPS_MIN) and every ~50 clean requests nudges it back up
    toward the configured ceiling. So an over-aggressive TM_HF_RPS self-corrects
    instead of failing — the setting sets the ceiling, the server decides the pace.
    """
    with _RL_LOCK:
        now = time.time()
        wait = _RL_NEXT[0] - now
        if wait > 0:
            time.sleep(wait)
        _RL_NEXT[0] = max(now, _RL_NEXT[0]) + 1.0 / max(0.5, _RL_RPS[0])


def _throttle_on_429():
    """Called on HTTP 429: halve the shared rate (floor _RL_RPS_MIN) so all threads slow."""
    with _RL_LOCK:
        _RL_429[0] += 1
        _RL_RPS[0] = max(_RL_RPS_MIN, _RL_RPS[0] / 2.0)
        return _RL_RPS[0]


def _throttle_on_ok():
    """Called on success: slowly recover toward the configured ceiling."""
    with _RL_LOCK:
        _RL_429[0] = max(0, _RL_429[0] - 1)
        if _RL_429[0] == 0:
            ceiling = float(os.environ.get("TM_HF_RPS", 8))
            _RL_RPS[0] = min(ceiling, _RL_RPS[0] * 1.05)


def ranged_get(url, a, b, tries=10, deadline=None):
    last = None
    for i in range(tries):
        if deadline is not None and time.time() > deadline:
            raise _Deadline()
        _throttle()
        try:
            r = _session().get(url, headers={**hf_headers(), "Range": f"bytes={a}-{b}"}, timeout=60)
            if r.status_code in (200, 206):
                if r.status_code == 200 and len(r.content) > (b - a + 1):  # server ignored Range
                    raise RuntimeError("Range ignored by server")
                _throttle_on_ok()
                return r.content
            last = f"HTTP {r.status_code}"
            if r.status_code == 429:
                # 429 = Too Many Requests. Shrink the shared rate (adaptive), honour
                # Retry-After, and push the shared clock forward so siblings slow too.
                rate = _throttle_on_429()
                ra = r.headers.get("Retry-After")
                try:
                    pause = float(ra) if ra is not None else 0.0
                except ValueError:
                    pause = 0.0
                pause = max(pause, min(120.0, 3.0 * (2 ** i)))
                with _RL_LOCK:
                    _RL_NEXT[0] = max(_RL_NEXT[0], time.time() + pause)
                time.sleep(pause + random.random() * 2.0)
                if i == 2:
                    log(f"  HF rate-limiting (429): shared rate -> {rate:.1f} req/s (auto-adapting)")
                continue
            if r.status_code in (500, 502, 503, 504):
                time.sleep(min(60.0, 2.0 * (2 ** i)) + random.random())
                continue
            time.sleep(min(30.0, 1.0 * (2 ** i)))
        except _Deadline:
            raise
        except Exception as e:  # noqa
            last = repr(e)
            time.sleep(min(30.0, 1.0 * (2 ** i)) + random.random())
    raise RuntimeError(f"range {a}-{b} failed after {tries} tries: {last}")


def make_ranged_reader(url, total, deadline=None, span=4096):
    """Small-read-optimised reader for parse_tar_headers.

    Header walks issue many 512-byte reads a few KB apart; HF resolve latency makes
    one-request-per-read prohibitively slow, so each read pulls a `span`-byte window
    and serves nearby reads from a one-window cache.
    """
    buf = {"start": -1, "data": b""}

    def read(a, n):
        end = min(a + n, total)
        if a >= total or end <= a:
            return b""
        if buf["start"] <= a and end <= buf["start"] + len(buf["data"]):
            o = a - buf["start"]
            return buf["data"][o:o + (end - a)]
        b = min(a + max(span, n) - 1, total - 1)
        data = ranged_get(url, a, b, deadline=deadline)
        buf["start"], buf["data"] = a, data
        return data[: end - a]

    return read


def _octal_or_b256(field):
    if field[0] & 0x80:
        return int.from_bytes(field, "big") & ((1 << (8 * len(field) - 1)) - 1)
    s = field.split(b"\0")[0].strip()
    return int(s, 8) if s else 0


def parse_tar_headers(read, total):
    """Walk a tar using read(offset, length)->bytes. Returns [(name, data_offset, size)]."""
    members, off, pending = [], 0, None
    while off + 512 <= total:
        h = read(off, 512)
        if len(h) < 512 or h.count(0) == 512:
            break
        name = h[:100].split(b"\0")[0].decode("utf8", "replace")
        if h[257:262] == b"ustar" and h[345:500].strip(b"\0"):
            name = h[345:500].split(b"\0")[0].decode("utf8", "replace") + "/" + name
        size = _octal_or_b256(h[124:136])
        flag = h[156:157]
        data_off = off + 512
        nxt = data_off + ((size + 511) // 512) * 512
        if flag in (b"L", b"x", b"g", b"K"):
            data = read(data_off, size)
            if flag == b"L":
                pending = data.split(b"\0")[0].decode("utf8", "replace")
            elif flag == b"x":
                for rec in data.split(b"\n"):
                    if b" path=" in rec:
                        pending = rec.split(b" path=", 1)[1].decode("utf8", "replace")
            off = nxt
            continue
        if flag in (b"0", b"\0"):
            members.append((pending or name, data_off, size))
        pending = None
        off = nxt
    return members


def build_index(deadline_ts, hard_deadline=None):
    """Index HF tars via range-request header walks. Resumable (index saved per tar).

    Tars are walked in a seeded-shuffled order and indexing stops early once the
    indexed shot bytes cover the download budget (x1.6 margin) - we only need enough
    archives to draw the subset, not the whole 500+ GB repo.

    `deadline_ts` is a SOFT deadline: once it passes, in-flight walks are cancelled
    when we already have enough (stop_reached) OR enough tars to draw a valid subset;
    otherwise indexing continues up to `hard_deadline` (the prep deadline), because an
    under-sized index risks the fatal <5-shot guard. Re-runs continue from the index.
    """
    hard_deadline = hard_deadline or deadline_ts
    from huggingface_hub import HfApi, hf_hub_url
    P = paths("random")
    os.makedirs(os.path.dirname(P["index"]), exist_ok=True)
    idx = json.load(open(P["index"])) if os.path.exists(P["index"]) else {"tars": {}, "shots": {}}
    api = HfApi()
    files = [f for f in api.list_repo_tree(CFG["repo_id"], repo_type="dataset", recursive=True)
             if getattr(f, "size", None) is not None and f.path.endswith(".tar")]
    files.sort(key=lambda f: f.path)
    log(f"HF repo lists {len(files)} tar files, {sum(f.size for f in files) / 1e9:.1f} GB total")
    indexed_bytes = sum(v[2] for v in idx["shots"].values())
    stop_gb = CFG["index_stop_gb"] or max(3.0, 1.6 * CFG["data_budget_gb"])
    min_tars = max(1, CFG["index_min_tars"])
    log(f"index: {len(idx['shots'])} shots / {indexed_bytes / 1e9:.1f} GB indexed (cached) across {len(idx['tars'])} tars; "
        f"stop submitting beyond {stop_gb:.1f} GB and >= {min_tars} tars")
    todo = [f for f in files if f.path not in idx["tars"]]
    random.Random(CFG["seed"]).shuffle(todo)  # seeded spread across the archive

    def stop_reached():
        return indexed_bytes >= stop_gb * 1e9 and len(idx["tars"]) >= min_tars

    def usable():
        # enough to select a valid subset even if a few tars still fail
        return len(idx["tars"]) >= min(3, min_tars)

    def work(f):
        url = hf_hub_url(CFG["repo_id"], f.path, repo_type="dataset")
        # reads may run until the HARD deadline; the loop cancels in-flight walks at the
        # soft deadline when the index is already usable.
        read = make_ranged_reader(url, f.size, deadline=hard_deadline)
        nreq = [0]
        mem = parse_tar_headers(lambda a, n: (nreq.__setitem__(0, nreq[0] + 1), read(a, n))[1], f.size)
        return f.path, url, mem

    with ThreadPoolExecutor(CFG["index_threads"]) as ex:
        it = iter(todo)
        pending = {}
        retries = {}
        inflight = set()

        def submit():
            if stop_reached():
                return False
            try:
                f = next(it)
            except StopIteration:
                # second pass: re-queue tars that failed transiently (e.g. HTTP 429),
                # never ones still being walked in another thread.
                cand = [f for f in todo
                        if f.path not in idx["tars"] and f.path not in inflight and retries.get(f.path, 0) < 3]
                if not cand:
                    return False
                f = cand[0]
                retries[f.path] = retries.get(f.path, 0) + 1
            fu = ex.submit(work, f)
            pending[fu] = f.path
            inflight.add(f.path)
            return True

        need = stop_gb * 1e9 - indexed_bytes
        cum, n_need = 0, 0
        if need > 0:
            for f in todo:
                cum += f.size
                n_need += 1
                if cum >= need:
                    break
        min_tars_left = max(0, min_tars - len(idx["tars"]))
        initial = min(CFG["index_threads"], max(n_need, min_tars_left), len(todo))
        log(f"submitting {initial} tars now (need ~{n_need} for budget, {min_tars_left} for archive spread)")
        for _ in range(initial):
            if not submit():
                break
        last_tick = time.time()
        while pending:
            now = time.time()
            if now > hard_deadline:
                log(f"index HARD deadline reached ({len(idx['tars'])} tars, {indexed_bytes / 1e9:.1f} GB): "
                    "cancelling remaining tar walks (download phase needs the rest)")
                for fu in pending:
                    fu.cancel()
                break
            if now > deadline_ts and (usable() or stop_reached()):
                log(f"index soft deadline reached with a usable index ({len(idx['tars'])} tars, "
                    f"{indexed_bytes / 1e9:.1f} GB): reserving the rest of prep for downloads")
                for fu in pending:
                    fu.cancel()
                break
            if now > deadline_ts and not usable():
                # too small to guarantee a valid subset -> keep going to the hard deadline
                if time.time() - last_tick > 120:
                    log(f"  index past soft deadline but only {len(idx['tars'])} tars: continuing to hard deadline")
                    last_tick = time.time()
            if stop_reached():
                log(f"enough archives indexed ({indexed_bytes / 1e9:.1f} GB across {len(idx['tars'])} tars): "
                    "cancelling remaining tar walks")
                for fu in pending:
                    fu.cancel()
                break
            try:
                done_fu = next(as_completed(list(pending), timeout=30), None)
            except TimeoutError:
                if time.time() - last_tick > 120:
                    log(f"  indexing... {len(pending)} tars in flight, {len(idx['tars'])} done, {indexed_bytes / 1e9:.1f} GB")
                    last_tick = time.time()
                continue
            fpath = pending.pop(done_fu)
            inflight.discard(fpath)
            try:
                path, url, mem = done_fu.result()
            except _Deadline:
                log(f"  {fpath}: walk aborted at indexing deadline (will resume next run)")
                continue
            except Exception as e:  # noqa
                log(f"  index FAILED for {fpath}: {repr(e)[:200]}")
                submit()
                continue
            n = 0
            for name, off, size in mem:
                base = os.path.basename(name)
                if not base.endswith(".zarr.zip"):
                    continue
                try:
                    sid = int(base[: -len(".zarr.zip")])
                except ValueError:
                    log("  skip unparsable member", name)
                    continue
                if str(sid) not in idx["shots"]:
                    idx["shots"][str(sid)] = [url, off, size]
                    indexed_bytes += size
                n += 1
            idx["tars"][path] = n
            log(f"  indexed {path}: {n} shots ({indexed_bytes / 1e9:.1f} GB total indexed)")
            json.dump(idx, open(P["index"], "w"))
            submit()
    missing = [f.path for f in files if f.path not in idx["tars"]]
    if missing and not stop_reached():
        log(f"WARNING: {len(missing)} tars not indexed (deadline/network?): {missing[:5]} - re-run prep to continue")
    log(f"index: {len(idx['shots'])} shots across {len(idx['tars'])} tars ({indexed_bytes / 1e9:.1f} GB)")
    return idx


def _quota_select(budget, official, avail, rng):
    """Random-priority selection of shots under a byte quota (order stays random)."""
    rng.shuffle(avail)
    used, chosen = 0, []
    for s in avail:
        sz = s[1]
        if used + sz > budget:
            continue  # keep scanning: smaller shots may still fit
        chosen.append(s[0])
        used += sz
    return chosen, used


def choose_shots(split, idx):
    from tokamark.data_split import read_data_split_csv
    csv, _, _ = split_files(split)
    tr, te, va = read_data_split_csv(csv)
    official = {"train": [int(s) for s in tr], "val": [int(s) for s in va], "test": [int(s) for s in te]}
    total = sum(len(v) for v in official.values())
    free = shutil.disk_usage(paths(split)["data"]).free
    budget = min(CFG["data_budget_gb"] * 1e9, max(0.0, free - 6e9))
    log(f"[{split}] official counts: " + ", ".join(f"{k}={len(v)}" for k, v in official.items()) + f" (total {total}); byte budget {budget / 1e9:.1f} GB")
    rng = random.Random(CFG["seed"])
    MIN_SHOTS = 5  # matches the fatal guard in cmd_prep
    sel, report = {}, {}
    for k, shots in official.items():
        avail = [(s, idx["shots"][str(s)][2]) for s in shots if str(s) in idx["shots"]]
        quota = budget * len(shots) / total
        rng.shuffle(avail)
        # guarantee >= MIN_SHOTS per split even when the byte quota is tiny: the 5
        # smallest available (cost-bounded), then fill the rest of the quota randomly.
        guard = sorted(avail, key=lambda t: t[1])[:MIN_SHOTS]
        guard_ids = {s for s, _ in guard}
        used1 = sum(sz for _, sz in guard)
        rest = [t for t in avail if t[0] not in guard_ids]
        extra, used2 = _quota_select(max(0.0, quota - used1), shots, rest, rng)
        chosen = [s for s, _ in guard] + extra
        used = used1 + used2
        sel[k] = chosen
        report[k] = dict(official=len(shots), in_archives=len(avail), selected=len(chosen), gb=round(used / 1e9, 2),
                         fraction_of_official=round(len(chosen) / max(1, len(shots)), 4))
    log(f"[{split}] selection: {json.dumps(report)}")
    return sel, report


def fetch_member(sid, meta, data_dir):
    url, off, size = meta
    final = os.path.join(data_dir, f"{sid}.zarr")
    if os.path.isdir(final):
        return sid, True, "cached"
    zpath = os.path.join(data_dir, f"{sid}.zip.part")
    tmpd = os.path.join(data_dir, f"{sid}.extract")
    try:
        with open(zpath, "wb") as fh:
            pos, CH = off, 64 * 2 ** 20
            while pos < off + size:
                end = min(pos + CH, off + size) - 1
                fh.write(ranged_get(url, pos, end))
                pos = end + 1
        if os.path.getsize(zpath) != size:
            raise RuntimeError("size mismatch")
        shutil.rmtree(tmpd, ignore_errors=True)
        with zipfile.ZipFile(zpath) as z:
            z.extractall(tmpd)
        root = tmpd
        if not (os.path.exists(os.path.join(root, "zarr.json")) or os.path.exists(os.path.join(root, ".zgroup"))):
            kids = [d for d in os.listdir(tmpd) if os.path.isdir(os.path.join(tmpd, d))]
            if len(kids) == 1 and (os.path.exists(os.path.join(tmpd, kids[0], "zarr.json")) or os.path.exists(os.path.join(tmpd, kids[0], ".zgroup"))):
                root = os.path.join(tmpd, kids[0])
            else:
                raise RuntimeError("no zarr root found in archive")
        import zarr
        zarr.open_group(root, mode="r")  # must open
        os.replace(root, final)
        return sid, True, "ok"
    except Exception as e:  # noqa
        return sid, False, repr(e)[:200]
    finally:
        shutil.rmtree(tmpd, ignore_errors=True)
        if os.path.exists(zpath):
            os.remove(zpath)


def cmd_prep(split):
    setup_imports()
    P = paths(split)
    os.makedirs(P["data"], exist_ok=True)
    os.makedirs(P["out"], exist_ok=True)
    deadline = T0 + CFG["prep_deadline_h"] * 3600
    # Indexing gets a soft sub-deadline (a fraction of the prep window) so a slow or
    # re-throttled index cannot starve downloads. But if the index is still short of the
    # size it needs when the soft deadline hits, build_index is allowed to use up to the
    # hard prep deadline instead - an under-sized index (which risks the fatal <5-shot
    # guard) is worse than a shorter download window. See build_index(soft, hard).
    index_frac = float(os.environ.get("TM_INDEX_TIME_FRAC", 0.25))
    index_soft = min(deadline, T0 + CFG["prep_deadline_h"] * 3600 * index_frac)
    idx = build_index(index_soft, hard_deadline=deadline)

    reuse = None
    if os.path.exists(P["sel"]) and not CFG["force_reselect"]:
        prev = json.load(open(P["sel"]))
        if prev.get("split") == split and min((len(v) for v in prev.get("shots", {}).values()), default=0) >= 5:
            reuse = prev
            log("reusing previous selection.json "
                f"({json.dumps({k: len(v) for k, v in reuse['shots'].items()})}); "
                "set TM_FORCE_RESELECT=1 to re-select shots")
    if reuse:
        sel, report = reuse["shots"], reuse["report"]
        for k in sel:
            report.setdefault(k, {})
            report[k]["official"] = report[k].get("official", len(sel[k]))
    else:
        sel, report = choose_shots(split, idx)

    # round-robin across splits so a deadline cut keeps the proportions; skip cached shots
    prep_splits = [s.strip() for s in os.environ.get("TM_PREP_SPLITS", "train,val,test").split(",") if s.strip()]
    order, lists = [], {k: [s for s in v if not os.path.isdir(os.path.join(P["data"], f"{s}.zarr"))]
                        for k, v in sel.items() if k in prep_splits}
    n_cached = sum(len(sel[k]) for k in prep_splits) - sum(len(lists[k]) for k in prep_splits)
    if n_cached:
        log(f"{n_cached} selected shots already on disk (cached)")
    while any(lists.values()):
        for k in prep_splits:
            if lists.get(k):
                order.append((k, lists[k].pop()))
    done, failed = set(), {}
    nworkers = int(os.environ.get("TM_FETCH_WORKERS", 4))
    attempts = {}
    inflight = set()
    with ThreadPoolExecutor(nworkers) as ex:
        it = iter(order)
        futs = {}
        def submit():
            try:
                k, s = next(it)
            except StopIteration:
                # re-queue shots that failed transiently (HTTP 429 etc.), bounded
                # retries, and never ones already in flight in another thread.
                cand = [(k, s) for (k, s) in order
                        if s not in done and s not in inflight and attempts.get(s, 0) < 3]
                if not cand:
                    return False
                k, s = cand[0]
                attempts[s] = attempts.get(s, 0) + 1
            futs[ex.submit(fetch_member, s, idx["shots"][str(s)], P["data"])] = (k, s)
            inflight.add(s)
            return True
        for _ in range(nworkers * 2):
            submit()
        while futs:
            fu = next(as_completed(list(futs)))
            k, s = futs.pop(fu)
            inflight.discard(s)
            sid, ok, msg = fu.result()
            if ok:
                done.add(sid)
                failed.pop(sid, None)
            else:
                failed[sid] = msg
            if len(done) % 25 == 0:
                log(f"  fetched {len(done)}/{len(order)} shots, failed {len(failed)}")
            if time.time() < deadline:
                submit()
        # one last pass for still-failed shots while time remains (paced = fewer 429s)
        for (k, s) in [t for t in order if t[1] not in done]:
            if time.time() >= deadline or s in done or attempts.get(s, 0) >= 3:
                continue
            attempts[s] = attempts.get(s, 0) + 1
            sid, ok, msg = fetch_member(s, idx["shots"][str(s)], P["data"])
            if ok:
                done.add(sid); failed.pop(sid, None)
            else:
                failed[sid] = msg
    final = {}
    for k, v in sel.items():
        if k in prep_splits:
            final[k] = [s for s in v if (s in done or os.path.isdir(os.path.join(P["data"], f"{s}.zarr")))]
        else:
            final[k] = v  # preserve the selected shot list for splits not being fetched
    if time.time() >= deadline and any(len(final[k]) < len(sel[k]) for k in prep_splits):
        log(f"prep deadline reached: using the {sum(len(final[k]) for k in prep_splits)} shots fetched so far "
            f"({sum(len(sel[k]) for k in prep_splits) - sum(len(final[k]) for k in prep_splits)} still missing)")
    report2 = {k: dict(report.get(k, {}), downloaded=len(final[k])) for k in final}
    meta = dict(split=split, shots=final, report=report2, failed=failed,
                config=dict(seed=CFG["seed"], task=CFG["task"], data_budget_gb=CFG["data_budget_gb"],
                            repo_id=CFG["repo_id"], git_revs=CFG["git_revs"], selected_at_min=round((time.time() - T0) / 60, 1)))
    json.dump(meta, open(P["sel"], "w"), indent=1)
    log(f"[{split}] FINAL shot counts: " + json.dumps({k: len(v) for k, v in final.items()}))
    if failed:
        log(f"  {len(failed)} shots failed and were excluded; first errors: {list(failed.items())[:3]}")
    req_splits = [k for k in prep_splits if k in final]
    if req_splits and min(len(final[k]) for k in req_splits) < 5:
        raise SystemExit(f"FATAL: fewer than 5 shots in some requested split ({req_splits}); refusing to train on a misleading subset. "
                         "Check network/index logs above, extend TM_DATA_GB / TM_PREP_DEADLINE_H, and re-run prep.")


# ======================================================================================
# Datasets / model (mirrors tokamark_baseline/run_training.py exactly, minus subset_of_shots)
# ======================================================================================
def build_data(split, which, shuffle_windows=True):
    setup_imports()
    import torch
    from torch.utils.data import DataLoader
    from tokamark.tasks import get_task_config, get_task_metadata
    from tokamark.tools.transforms.compose_transform import ComposeTransforms
    from tokamark.data import initialize_MAST_dataset, initialize_TokaMark_dataset
    from src.model_transform import ModelTransform_1, ModelTransform_2
    from src.trainer import model_collate_fn
    from utils import seed_worker

    P = paths(split)
    sel = json.load(open(P["sel"]))
    assert sel["split"] == split, "selection file belongs to another split protocol!"
    _, stats, outl = split_files(split)
    patch_stats_module(stats)
    cfgt = get_task_config(task_name=CFG["task"])
    cfgt["stride_window"] = 0.005  # same override as run_training.py for task_1-x/2-x
    meta = get_task_metadata(config_task=cfgt, verbose=False)
    tf = ComposeTransforms([ModelTransform_1(meta | cfgt), ModelTransform_2(meta | cfgt)])
    nw = CFG["dl_workers"]
    if nw < 0:
        nw = max(2, min(8, os.cpu_count() or 2))
    smgr = {"base_local_zarr_path": P["data"]}
    loaders, g = {}, torch.Generator()
    g.manual_seed(CFG["seed"])
    # Tensor IPC must not go through /dev/shm (Docker default is 64 MB; a 512-window
    # batch deadlocks the workers silently). file_system uses /tmp instead.
    torch.multiprocessing.set_sharing_strategy("file_system")
    for w in which:
        mast = initialize_MAST_dataset(config_task=cfgt, shots_list=sel["shots"][w], local_flag=True, use_std_scaling=True,
                                       stats_metadata_file_path=stats, use_nan_filling=False, remove_outliers=True,
                                       outlier_metadata_file=outl, remove_bad_efit_rating=True,
                                       store_manager_settings=smgr, verbose=False)
        kw = dict(test_mode=True)
        if w == "test":
            kw["shuffle_windows"] = False
        ds = initialize_TokaMark_dataset(dataset=mast, task_metadata=meta, config_metadata=cfgt, custom_transform=tf, **kw)
        if ds is None:
            raise RuntimeError(f"{w} dataset empty after official filtering (outliers/bad EFIT)")
        # TokaMarkDataset is an IterableDataset (no __getitem__), so probe via the
        # iterator: time how long one window takes to materialise. A real failure here
        # (empty dataset, bad zarr) is fatal, not a harmless warning.
        t_probe = time.time()
        first = next(iter(ds), None)
        dt = time.time() - t_probe
        if first is None:
            raise RuntimeError(f"{w} dataset yielded no windows after official filtering "
                               f"(outliers/bad EFIT / empty shots) - cannot train on it")
        est = dt * CFG["batch_size"] / max(1, min(max(nw, 1), CFG["batch_size"]))
        log(f"{w} dataset: {len(sel['shots'][w])} shots ({type(ds).__name__}, streaming windows); "
            f"first window took {dt:.2f}s "
            f"(=> first batch of {CFG['batch_size']} will take roughly {est / 60:.1f} min)")
        extra = dict(drop_last=True) if w == "train" else {}
        if nw > 0:
            extra["persistent_workers"] = True
        loaders[w] = DataLoader(ds, batch_size=CFG["batch_size"], num_workers=nw, collate_fn=model_collate_fn,
                                worker_init_fn=seed_worker, generator=g, pin_memory=torch.cuda.is_available(),
                                prefetch_factor=2 if nw > 0 else None, **extra)
        log(f"{w} loader ready: {len(sel['shots'][w])} shots, batch_size={CFG['batch_size']}, workers={nw}")
    return loaders, meta | cfgt


def make_model(loader, meta):
    model_type = os.environ.get("TM_MODEL", "mlp").lower().strip()
    if model_type == "lstm":
        from src.multi_conv_lstm_model import create_lstm_architecture
        return create_lstm_architecture(dataloader_=loader, dict_metadata=meta, verbose=False)
    else:
        from src.multi_conv_mlp_model import create_cnn_architecture
        return create_cnn_architecture(dataloader_=loader, dict_metadata=meta, verbose=False)


# ======================================================================================
# TRAIN: resumable, deadline-aware, optional DataParallel (benchmarked)
# ======================================================================================
def atomic_save(obj, path):
    import torch
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def pick_parallel(model, loader, dev, criterion, move):
    """Return (module_to_call, label). Benchmarks 1 GPU vs DataParallel on one cached batch."""
    import torch, torch.nn as nn
    n = torch.cuda.device_count()
    if n < 2 or CFG["use_dp"] == "0":
        return model, f"single GPU ({n} visible)"
    if CFG["use_dp"] == "1":
        return nn.DataParallel(model), f"DataParallel x{n} (forced)"
    try:
        batch = first_batch_timed(loader, tag="benchmark batch")
        x, y = move(batch, dev)
        def bench(m):
            opt = torch.optim.SGD(m.parameters(), lr=0.0)  # lr 0: benchmark does not change weights
            m.train()
            for i in range(14):
                if i == 4:
                    torch.cuda.synchronize(); t = time.time()
                opt.zero_grad(); criterion(m(*x), y).backward(); opt.step()
            torch.cuda.synchronize()
            return (time.time() - t) / 10
        sd = {k: v.clone() for k, v in model.state_dict().items()}
        t1 = bench(model)
        dp = nn.DataParallel(model)
        t2 = bench(dp)
        model.load_state_dict(sd)
        log(f"GPU benchmark (compute only, same batch): 1 GPU {t1 * 1e3:.0f} ms/step, DataParallel {t2 * 1e3:.0f} ms/step")
        if t2 < t1 / 1.15:
            return dp, f"DataParallel x{n} (benchmark {t1 / t2:.2f}x faster)"
        return model, "single GPU (DataParallel not clearly faster)"
    except Exception as e:  # noqa
        log("parallel benchmark failed, using single GPU:", repr(e)[:200])
        return model, "single GPU (benchmark error)"


def cmd_train(split):
    setup_imports()
    import torch
    from src.trainer import MultiOutputMSELoss, move_batch_to_device
    from utils import set_seed
    set_seed(CFG["seed"])
    P = paths(split)
    os.makedirs(P["model_dir"], exist_ok=True)
    sel = json.load(open(P["sel"]))
    run_config = dict(task=CFG["task"], split=split, seed=CFG["seed"], batch_size=CFG["batch_size"],
                      lr=CFG["lr"], weight_decay=CFG["weight_decay"], max_steps=CFG["max_steps"],
                      validate_every=CFG["validate_every"], patience=CFG["patience"],
                      train_deadline_h=CFG["train_deadline_h"], use_dp=CFG["use_dp"],
                      git_revs=CFG["git_revs"],
                      shots={k: len(v) for k, v in sel["shots"].items()},
                      report=sel["report"])
    json.dump(run_config, open(os.path.join(P["out"], "run_config.json"), "w"), indent=1)
    log(f"run_config: {json.dumps(run_config)}")
    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    log(f"device={dev}, gpus={torch.cuda.device_count()}, cpus={os.cpu_count()}")
    phase = {"p": "building dataloaders", "step": 0}

    def _heartbeat():
        while True:
            time.sleep(60)
            log(f"  heartbeat: {phase['p']} (step {phase['step']})")

    threading.Thread(target=_heartbeat, daemon=True).start()
    loaders, meta = build_data(split, ["train", "val"])
    phase["p"] = "building model"
    t_model = time.time()
    log("building model (shape inference walks the dataset for one sample)...")
    model = make_model(loaders["train"], meta).to(dev)
    log(f"model built in {time.time() - t_model:.1f}s, "
        f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")
    crit = MultiOutputMSELoss()

    # ---- fail-fast smoke check (a few seconds): shapes, NaNs, forward/backward ----
    phase["p"] = "first batch (smoke)"
    b = first_batch_timed(loaders["train"], tag="first train batch")
    sid, wid, xs, ys = b
    x, y = move_batch_to_device(b, dev)
    out = model(*x)
    log("smoke: input shapes", [tuple(t.shape) for t in x], "| target shapes", [tuple(t.shape) for t in y],
        "| output shapes", [tuple(o.shape) for o in out])
    assert all(o.shape == t.shape for o, t in zip(out, y)), "output/target shape mismatch"
    assert all(torch.isfinite(t).all() for t in x), "non-finite inputs after collate"
    l = crit(out, y); l.backward()
    assert torch.isfinite(l), "non-finite initial loss"
    log(f"smoke OK: initial loss {l.item():.4f}")
    model.zero_grad(set_to_none=True)

    opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3, min_lr=1e-6)
    st = dict(step=0, best=float("inf"), no_improve=0, epoch=0, oom=0,
              hist=dict(step=[], train_loss=[], val_loss=[], lr=[], t=[]))
    last = os.path.join(P["model_dir"], "last.pt")
    if os.path.exists(last) and not CFG["train_from_scratch"]:
        ck = torch.load(last, map_location=dev, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); st = ck["state"]
        if "sched" in ck and ck["sched"] is not None:
            sched.load_state_dict(ck["sched"])
        st.setdefault("oom", 0)
        st["hist"].setdefault("t", [])
        st["hist"].setdefault("lr", [])
        log(f"RESUMED from step {st['step']} (best val {st['best']:.5f})")
    elif CFG["train_from_scratch"] and os.path.exists(last):
        log("TM_TRAIN_FROM_SCRATCH=1: ignoring existing checkpoint, training from step 0")
    runner, label = pick_parallel(model, loaders["train"], dev, crit, move_batch_to_device) if st["step"] == 0 else (model, "single GPU (resumed)")
    log("parallel mode:", label)

    @torch.no_grad()
    def validate():
        phase["p"] = "validation"
        runner.eval(); tot = n = 0
        for vb in loaders["val"]:
            if vb is None: continue
            vx, vy = move_batch_to_device(vb, dev)
            lv = crit(runner(*vx), vy); bs = vy[0].shape[0]
            tot += lv.item() * bs; n += bs
        phase["p"] = "training loop"
        if n == 0:
            raise RuntimeError("validation loader produced zero samples - check selection.json / val shots")
        return tot / n

    def save_last():
        sched_st = sched.state_dict() if sched is not None else None
        atomic_save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched_st, state=st), last)
        atomic_save(st["hist"], os.path.join(P["model_dir"], "training_history.pt"))
        json.dump(st["hist"], open(os.path.join(P["model_dir"], "training_history.json"), "w"))

    deadline = T0 + CFG["train_deadline_h"] * 3600
    phase["p"] = "training loop"
    it, run_loss, errs, bad, oom, t_last_ck, t_tr = iter(loaders["train"]), [], 0, 0, 0, time.time(), time.time()
    stop_reason = "max_steps"
    while st["step"] < CFG["max_steps"]:
        if time.time() > deadline:
            stop_reason = "time deadline"; break
        phase["p"] = f"waiting for batch (step {st['step']})"
        try:
            batch = next(it)
        except StopIteration:
            it = iter(loaders["train"]); st["epoch"] += 1; run_loss = []; log(f"epoch {st['epoch']} done"); continue
        except Exception as e:  # noqa  (e.g. a corrupt shot inside a worker)
            errs += 1; log(f"data error #{errs}: {repr(e)[:200]}")
            if errs > 20: raise
            it = iter(loaders["train"]); continue
        if batch is None: continue
        phase["p"] = f"train step {st['step'] + 1}"
        try:
            runner.train()
            x, y = move_batch_to_device(batch, dev)
            opt.zero_grad(set_to_none=True)
            loss = crit(runner(*x), y)
            if not torch.isfinite(loss):
                bad += 1; log(f"non-finite loss, skipping (#{bad})")
                if bad > 200: raise RuntimeError("too many non-finite losses")
                continue
            bad = 0
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
        except torch.cuda.OutOfMemoryError:
            oom += 1; st["oom"] = oom
            torch.cuda.empty_cache()
            log(f"CUDA OOM at step {st['step']} (#{oom}) - batch skipped; current batch_size={CFG['batch_size']}")
            if oom >= 3:
                raise RuntimeError(
                    f"CUDA out of memory {oom}x with batch_size={CFG['batch_size']} on this GPU. "
                    "Re-run the train cell with a smaller TM_BATCH in the config cell (e.g. 256 or 128); "
                    f"training will resume from {last} without losing progress.")
            continue
        st["step"] += 1; run_loss.append(loss.item()); phase["step"] = st["step"]
        if st["step"] <= 10 or st["step"] % 20 == 0:
            el = time.time() - t_tr; t_tr = time.time()
            log(f"step {st['step']} loss {sum(run_loss) / len(run_loss):.4f} "
                f"({(CFG['batch_size'] if st['step'] <= 10 else 20 * CFG['batch_size']) / el:.0f} samples/s)")
        if st["step"] == 1:
            save_last()  # keep progress even if the session dies early
        if st["step"] % CFG["validate_every"] == 0:
            v = validate(); h = st["hist"]
            cur_lr = opt.param_groups[0]["lr"]
            for k, val in zip(("step", "train_loss", "val_loss", "lr", "t"), (st["step"], sum(run_loss) / len(run_loss), v, cur_lr, time.time() - T0)):
                h[k].append(val)
            sched.step(v)
            if v < st["best"]:
                st["best"], st["no_improve"] = v, 0
                atomic_save(model.state_dict(), os.path.join(P["model_dir"], "best_model.pt"))
            else:
                st["no_improve"] += 1
            log(f"VAL step {st['step']} val_loss {v:.5f} best {st['best']:.5f} no_improve {st['no_improve']}/{CFG['patience']}")
            save_last(); t_last_ck = time.time()
            if st["no_improve"] >= CFG["patience"]:
                stop_reason = "early stopping (official patience rule)"; break
        elif time.time() - t_last_ck > CFG["ckpt_every_s"]:
            save_last(); t_last_ck = time.time()
    v = validate()
    if v < st["best"] or not os.path.exists(os.path.join(P["model_dir"], "best_model.pt")):
        st["best"] = v; atomic_save(model.state_dict(), os.path.join(P["model_dir"], "best_model.pt"))
    save_last()
    summary = dict(stop_reason=stop_reason, steps=st["step"], epochs=st["epoch"], best_val=st["best"],
                   parallel=label, cuda_oom_events=st.get("oom", 0), data_errors=errs,
                   duration_min=round((time.time() - T0) / 60, 1),
                   model_dir=P["model_dir"], **{k: run_config[k] for k in ("task", "split", "seed", "batch_size", "lr",
                                                                           "weight_decay", "validate_every", "patience",
                                                                           "git_revs", "shots", "report")})
    json.dump(summary, open(os.path.join(P["out"], "train_summary.json"), "w"), indent=1)
    log(f"TRAIN DONE ({stop_reason}) at step {st['step']}, best val {st['best']:.5f}; summary -> train_summary.json")


# ======================================================================================
# EVAL: official evaluator on test shots, un-standardized NRMSE
# ======================================================================================
def cmd_eval(split):
    setup_imports()
    import torch
    from tokamark.evaluator import WindowMetricsAccumulator, compute_metrics
    from src.evaluator import cnn_unstd_evaluation_per_shot
    P = paths(split)
    os.makedirs(P["results"], exist_ok=True)
    if not os.path.exists(os.path.join(P["model_dir"], "best_model.pt")):
        raise SystemExit(f"FATAL: {P['model_dir']}best_model.pt not found - run train first")
    loaders, meta = build_data(split, ["test"])
    model = make_model(loaders["test"], meta)
    acc = WindowMetricsAccumulator(CFG["task"])
    cnn_unstd_evaluation_per_shot(test_dataloader=loaders["test"], config_task=meta, cnn_model=model,
                                  output_dir=P["model_dir"], accumulator=acc)
    compute_metrics(task=CFG["task"], output_dir=P["results"], window_metrics_accumulator=acc,
                    save_windows_metrics=True, save_task_metrics=True)
    sel = json.load(open(P["sel"]))
    log(f"[{split}] EVAL DONE. Results in {P['results']}")
    log(f"[{split}] shot counts used: {json.dumps(sel['report'])}")
    for f in sorted(os.listdir(P["results"])):
        if f.endswith((".json", ".yaml", ".yml")):
            print(f"--- {f}\n" + open(os.path.join(P["results"], f)).read()[:3000])
    for root, _, files in os.walk(P["results"]):
        for f in sorted(files):
            if f.endswith(".csv"):
                print(f"--- {os.path.join(root, f)}")
                with open(os.path.join(root, f)) as fh:
                    for i, line in enumerate(fh):
                        if i > 12: print("  ..."); break
                        print("  " + line.rstrip())


# ======================================================================================
# PLOT: history curves + optional reconstruction-quality viz (never blocks core results)
# ======================================================================================
def cmd_plot(split):
    setup_imports()
    import torch
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    P = paths(split)
    os.makedirs(P["figures"], exist_ok=True)

    # (a) training/validation curves - core, from saved history
    hp = os.path.join(P["model_dir"], "training_history.pt")
    try:
        h = torch.load(hp, map_location="cpu", weights_only=False)
        steps, tr_, va_ = h["step"], h["train_loss"], h["val_loss"]
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(steps, tr_, label="train loss (running mean at validation)")
        ax.plot(steps, va_, label="validation loss")
        ax.set_xlabel("step"); ax.set_ylabel("MultiOutputMSELoss (standardised)")
        ax.set_title(f"TokaMark {CFG['task']} baseline - {split} split, seed {CFG['seed']}")
        ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
        fig.savefig(os.path.join(P["figures"], "loss_curves.png"), dpi=120)
        plt.close(fig)
        log(f"saved loss_curves.png ({len(steps)} validation points)")
    except Exception as e:  # noqa
        log("loss-curve plot failed:", repr(e)[:200])

    # (b) reconstruction-quality viz: predicted vs true psi on a few test windows
    try:
        from src.trainer import move_batch_to_device
        from src.evaluator import cnn_unstd_evaluation_per_shot  # noqa: F401 (kept for parity)
        dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        loaders, meta = build_data(split, ["test"])
        model = make_model(loaders["test"], meta).to(dev)
        model.load_state_dict(torch.load(os.path.join(P["model_dir"], "best_model.pt"), map_location=dev, weights_only=False))
        model.eval()
        batch = first_batch(loaders["test"])
        # move_batch_to_device returns only (inputs, targets); shot/window ids stay on the
        # raw batch tuple (sid, wid, xs, ys), so read them from there.
        raw = batch
        x, y = move_batch_to_device(batch, dev)
        shot_ids, win_ids = raw[0], raw[1]
        with torch.no_grad():
            y_pred = model(*x)
        mstd = meta["output"]["equilibrium-psi"]
        mu, sd = float(mstd["mean"]), float(mstd["std"])
        n = min(CFG["plot_windows"], y[0].shape[0])
        fig, axes = plt.subplots(n, 3, figsize=(11, 3.2 * n))
        if n == 1:
            axes = [axes]
        for i in range(n):
            true = (y[0][i, 0].cpu().numpy() * sd + mu)
            pred = (y_pred[0][i, 0].cpu().numpy() * sd + mu)
            err = abs(pred - true)
            mae_w = float(err.mean())
            for j, (img, ttl) in enumerate(((true, "True psi"), (pred, "Pred psi"), (err, f"|Error| (MAE: {mae_w:.4f} Wb)"))):
                im = axes[i][j].imshow(img, cmap="viridis" if j < 2 else "magma")
                axes[i][j].set_title(f"{ttl} (shot {int(shot_ids[i])})", fontsize=9)
                axes[i][j].axis("off")
                fig.colorbar(im, ax=axes[i][j], fraction=0.046)
        fig.tight_layout()
        fig.savefig(os.path.join(P["figures"], "psi_reconstruction_samples.png"), dpi=120)
        plt.close(fig)
        log("saved psi_reconstruction_samples.png")
    except Exception as e:  # noqa
        log("optional psi reconstruction viz failed (core results unaffected):", repr(e)[:300])
    log(f"figures in {P['figures']}")


# ======================================================================================
# SUMMARY: one human-readable recap of the whole run
# ======================================================================================
def cmd_summary(split):
    setup_imports()
    import csv as _csv
    P = paths(split)
    lines = [f"# TokaMark {CFG['task']} baseline run - {split} split", ""]
    def add_json(fname, title):
        p = os.path.join(P["out"], fname)
        if os.path.exists(p):
            lines.append(f"## {title}")
            lines.append("```json")
            lines.append(json.dumps(json.load(open(p)), indent=1))
            lines.append("```")
        else:
            lines.append(f"## {title}\n\nMISSING: {p}")
        lines.append("")
    add_json("selection.json", "Data subset & split (official membership, budget-limited selection)")
    add_json("run_config.json", "Run configuration")
    add_json("train_summary.json", "Training summary")
    lines.append("## Evaluation (official evaluator, un-standardised NRMSE)")
    tm = os.path.join(P["results"], CFG["task"], "task_metrics.csv")
    if os.path.exists(tm):
        rows = list(_csv.DictReader(open(tm)))
        for r in rows:
            name = r["feature_name"]
            lines.append(f"- **{name}**: NRMSE={float(r['NRMSE_mean']):.4f} (+/-{float(r['NRMSE_std_pop']):.4f}), "
                         f"RMSE={float(r['RMSE_mean']):.5f}, MAE={float(r['MAE_mean']):.5f}, n_shots={r['n_shots']}")
    else:
        lines.append(f"MISSING: {tm} (eval not run?)")
    lines.append("")
    lines.append("## Artifacts")
    for label, p in (("selection", P["sel"]), ("run config", os.path.join(P["out"], "run_config.json")),
                     ("train summary", os.path.join(P["out"], "train_summary.json")),
                     ("best model", os.path.join(P["model_dir"], "best_model.pt")),
                     ("resume checkpoint", os.path.join(P["model_dir"], "last.pt")),
                     ("history", os.path.join(P["model_dir"], "training_history.pt")),
                     ("results dir", P["results"]), ("figures dir", P["figures"])):
        lines.append(f"- {label}: `{p}` {'OK' if os.path.exists(p) else '**MISSING**'}")
    text = "\n".join(lines)
    with open(os.path.join(P["out"], "summary.md"), "w") as fh:
        fh.write(text)
    print(text)
    log(f"summary -> {os.path.join(P['out'], 'summary.md')}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["prep", "train", "eval", "plot", "summary"])
    ap.add_argument("--split", default="random", choices=["random", "temporal"])
    a = ap.parse_args()
    {"prep": cmd_prep, "train": cmd_train, "eval": cmd_eval, "plot": cmd_plot, "summary": cmd_summary}[a.cmd](a.split)


In [ ]:
%%writefile /kaggle/working/robustness.py
#!/usr/bin/env python
"""TokaMark task_1-3 sensor-robustness harness: corrupt | eval-robust | report.

Stress-tests a *trained, frozen* task_1-3 model under the six sensor-failure
scenarios of Gupta (arXiv 2607.11915), on the TEST split only, with optional
imputation. Reuses the exact pipeline helpers (build_data / make_model) and the
official un-standardise + hierarchical-metric protocol so the CLEAN baseline
reproduces the eval-time NRMSE before any corruption is trusted.

Commands:
  python robustness.py corrupt     --split random     # build corrupted test loaders + manifest
  python robustness.py eval-robust --split random     # run the sweep, save metrics
  python robustness.py report      --split random     # figures + robustness.md

All settings come from TM_* env vars (same CFG contract as pipeline.py). Robustness
knobs:
  TM_ROB_SCENARIOS  comma list; default: random_dropout,channel_ablation,category_abl,
                    temporal_gap,group_failure,disruption_proximate  (or 'all')
  TM_ROB_IMPUTE     zero,mean,ffill (comma list); default: zero,mean,ffill
  TM_ROB_SEED       RNG seed for corruption (default TM_SEED)
  TM_ROB_CLEAN_ONLY 1 -> only the clean pass (sanity check), skip corruption
  TM_ROB_BATCH      forward batch size (default TM_BATCH)
"""
import os, sys, json, time, math, argparse
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pipeline as P
from pipeline import CFG, T0, log, setup_imports, build_data, make_model

TASK = CFG["task"]

SIGNALS = [
    "magnetics/flux_loop_flux",
    "magnetics/b_field_pol_probe_ccbv_field",
    "magnetics/b_field_pol_probe_obr_field",
    "magnetics/b_field_pol_probe_obv_field",
    "magnetics/b_field_tor_probe_saddle_voltage",
    "pf_active/coil_current",
    "pf_active/solenoid_current",
    "summary/ip",
]
SIGNAL_SHORT = ["flux_loop", "ccbv", "obr", "obv", "saddle_v", "coil_i", "solenoid_i", "ip"]

CATEGORY_GROUPS = {
    "flux_loop": [0],
    "pickup_fields": [1, 2, 3],
    "saddle_voltage": [4],
    "active_coils": [5],
    "solenoid": [6],
    "plasma_current": [7],
}
CORRELATED_GROUPS = {
    "magnetic_probes": [1, 2, 3],
    "power_supply": [5, 6],
    "magnetics_all": [0, 1, 2, 3, 4],
}

SCENARIOS = ["random_dropout", "channel_ablation", "category_abl",
             "temporal_gap", "group_failure", "disruption_proximate"]
SEVERITIES = {
    "random_dropout": [0.10, 0.25, 0.50],
    "channel_ablation": [1, 3, 6],
    "category_abl": [1],
    "temporal_gap": [0.20, 0.40, 0.60],
    "group_failure": [1],
    "disruption_proximate": [0.10, 0.25, 0.50],
}
IMPUTE_MODES = ("zero", "mean", "ffill")


def envf(k, d):
    return type(d)(os.environ.get(k, d))


def rob_cfg():
    scen = os.environ.get("TM_ROB_SCENARIOS", "all")
    if scen.strip() == "all":
        scen = SCENARIOS
    else:
        scen = [s.strip() for s in scen.split(",") if s.strip()]
    imp = [m.strip() for m in os.environ.get("TM_ROB_IMPUTE", "zero,mean,ffill").split(",") if m.strip()]
    bad = [s for s in scen if s not in SCENARIOS]
    bad += [m for m in imp if m not in IMPUTE_MODES]
    if bad:
        raise SystemExit(f"FATAL: unknown robustness options: {bad}")
    return dict(
        scenarios=scen,
        imputes=imp,
        seed=envf("TM_ROB_SEED", CFG["seed"]),
        clean_only=os.environ.get("TM_ROB_CLEAN_ONLY", "0") == "1",
        batch=envf("TM_ROB_BATCH", CFG["batch_size"]),
    )


def rob_paths(split):
    out = os.path.join(CFG["work"], "out", split)
    rdir = os.path.join(out, "robustness")
    return dict(
        dir=rdir,
        manifest=os.path.join(rdir, "robust_manifest.json"),
        metrics=os.path.join(rdir, "robust_metrics.json"),
        windows=os.path.join(rdir, "robust_windows.csv"),
        figures=os.path.join(rdir, "figures"),
        report=os.path.join(rdir, "robustness.md"),
        model_dir=os.path.join(out, "model") + "/",
    )


# ======================================================================================
# CORRUPTION PRIMITIVES  (operate on a per-channel torch tensor x: (B, W, C, L))
# ======================================================================================
def _as_np(t):
    return t.detach().cpu().numpy() if hasattr(t, "detach") else np.asarray(t)


def _mask_random_dropout(arr, sev, rng):
    """Zero a fraction `sev` of the *finite, non-zero* values (Gupta's definition).

    Returns (masked_array, hole_mask) where hole_mask marks exactly the values we zeroed.
    """
    m = np.isfinite(arr) & (arr != 0)
    idx = np.flatnonzero(m.ravel())
    out = arr.copy()
    holes = np.zeros_like(arr, dtype=bool)
    if idx.size == 0:
        return out, holes
    k = int(round(sev * idx.size))
    drop = rng.choice(idx, size=min(k, idx.size), replace=False)
    out.ravel()[drop] = 0.0
    holes.ravel()[drop] = True
    return out, holes


def _mask_temporal_gap(arr, frac, rng, where="random"):
    """Zero a contiguous fraction `frac` of the time axis (last axis)."""
    out = arr.copy()
    holes = np.zeros_like(arr, dtype=bool)
    L = arr.shape[-1]
    g = max(1, int(round(frac * L)))
    if where == "front":
        s = 0
    elif where == "end":
        s = L - g
    else:
        s = int(rng.integers(0, max(1, L - g + 1)))
    out[..., s:s + g] = 0.0
    holes[..., s:s + g] = True
    return out, holes


def _zero_channels(arr, chans):
    """Zero whole channels; return (masked, holes) with holes marking the zeroed area."""
    out = arr.copy()
    holes = np.zeros_like(arr, dtype=bool)
    for c in chans:
        out[..., c, :] = 0.0
        holes[..., c, :] = True
    return out, holes


def _impute(masked, holes, mode, channel_mean=0.0):
    """Fill ONLY the positions flagged in `holes`. mode in {zero,mean,ffill}.

    'zero'   -> leave the zeros (TokaMark default; no filling).
    'mean'   -> put `channel_mean` (TRAIN-split signal mean) in every hole.
    'ffill'  -> carry the last valid (non-hole) value forward along time, per channel.
    """
    if mode == "zero":
        return masked
    out = np.array(masked, dtype=masked.dtype, copy=True)
    if not holes.any():
        return out
    if mode == "mean":
        out[holes] = float(channel_mean)
        return out
    if mode == "ffill":
        ow = out.reshape(-1, out.shape[-2], out.shape[-1])
        hw = holes.reshape(-1, holes.shape[-2], holes.shape[-1])
        for i in range(ow.shape[0]):
            for c in range(ow.shape[1]):
                row, hole = ow[i, c], hw[i, c]
                last = 0.0
                for t in range(row.shape[0]):
                    if hole[t]:
                        row[t] = last
                    else:
                        last = row[t]
        return out
    raise ValueError(mode)


# ======================================================================================
# SCENARIO APPLICATION  (returns a new inputs list, corruption bookkeeping dict)
# ======================================================================================
def apply_scenario(inputs, scenario, sev, impute, train_means, rng):
    """inputs: list of 8 torch tensors (B, W, C, L). Returns (corrupted_inputs, info).

    Corruption is applied on CPU in numpy, then re-wrapped to torch tensors matching
    each input's original dtype. Imputation 'mean' uses `train_means` (per signal, a
    scalar computed from the TRAIN split) so nothing about the test split leaks.
    """
    import torch
    info = dict(scenario=scenario, severity=sev, impute=impute, details={})
    # channel-set scenarios are deterministic per task (draw once, not per channel)
    ablate_chans = None
    if scenario == "channel_ablation":
        n = min(int(sev), len(SIGNALS))
        ablate_chans = sorted(rng.choice(len(SIGNALS), size=n, replace=False).tolist())
        info["details"]["channels_zeroed"] = [SIGNAL_SHORT[c] for c in ablate_chans]
    out = []
    for i, t in enumerate(inputs):
        arr = _as_np(t).astype(np.float64)
        masked, holes = arr, np.zeros_like(arr, dtype=bool)
        if scenario == "random_dropout":
            masked, holes = _mask_random_dropout(arr, float(sev), rng)
        elif scenario == "channel_ablation":
            if i in ablate_chans:
                masked, holes = _zero_channels(arr, [0])
        elif scenario == "temporal_gap":
            masked, holes = _mask_temporal_gap(arr, float(sev), rng, where="random")
        elif scenario == "disruption_proximate":
            masked, holes = _mask_temporal_gap(arr, float(sev), rng, where="end")
        else:
            raise ValueError(scenario)
        masked = _impute(masked, holes, impute, channel_mean=train_means.get(i, 0.0))
        out.append(torch.as_tensor(masked, dtype=getattr(t, "dtype", None)))
    return out, info


def apply_group(inputs, scenario, sev, impute, train_means, rng, group_key, group_chans):
    """Like apply_scenario but zeroes a fixed set of channels (category / group runs)."""
    import torch
    group_chans = [c for c in group_chans if 0 <= c < len(inputs)]
    info = dict(scenario=scenario, severity=sev, impute=impute,
                details={"group": group_key, "channels": [SIGNAL_SHORT[c] for c in group_chans]})
    out = []
    for i, t in enumerate(inputs):
        arr = _as_np(t).astype(np.float64)
        if i in group_chans:
            masked, holes = _zero_channels(arr, [0])
        else:
            masked, holes = arr, np.zeros_like(arr, dtype=bool)
        masked = _impute(masked, holes, impute, channel_mean=train_means.get(i, 0.0))
        out.append(torch.as_tensor(masked, dtype=getattr(t, "dtype", None)))
    return out, info


# ======================================================================================
# METRICS  (reuse the official hierarchy: per-window -> per-shot -> across shots)
# ======================================================================================
def _psi_mean_std(meta):
    m = meta["output"]["equilibrium-psi"]
    return float(m["mean"]), float(m["std"])


def window_metrics(y_true, y_pred):
    """y_true/y_pred: (N, 65, 65) numpy, real units. Returns per-window RMSE, MAE arrays."""
    diff = y_pred - y_true
    rmse = np.sqrt(np.nanmean(diff ** 2, axis=(1, 2)))
    mae = np.nanmean(np.abs(diff), axis=(1, 2))
    return rmse, mae


def aggregate(shot_ids, rmse, mae, std):
    """Per-window -> per-shot mean -> across-shot mean/std; plus window-level stats.

    Mirrors tokamark.evaluator: RMSE per window, then averaged per shot, then NRMSE =
    shot_RMSE / output_std, then mean/std across shots (population sd, ddof=0).
    """
    shot_ids = np.asarray(shot_ids, dtype=np.int64)
    rmse = np.asarray(rmse, dtype=np.float64)
    mae = np.asarray(mae, dtype=np.float64)
    uniq = np.unique(shot_ids)
    shot_rmse, shot_mae = [], []
    for s in uniq:
        m = shot_ids == s
        # Official TokaMark: shot RMSE = sqrt(mean(window_RMSE ** 2))
        shot_rmse.append(float(np.sqrt(np.nanmean(rmse[m] ** 2))))
        shot_mae.append(float(np.nanmean(mae[m])))
    shot_rmse = np.asarray(shot_rmse)
    shot_mae = np.asarray(shot_mae)
    return dict(
        n_shots=int(len(uniq)),
        n_windows=int(len(rmse)),
        NRMSE=float(np.nanmean(shot_rmse / std)),
        NRMSE_std=float(np.nanstd(shot_rmse / std, ddof=0)),
        NMAE=float(np.nanmean(shot_mae / std)),
        NMAE_std=float(np.nanstd(shot_mae / std, ddof=0)),
        RMSE=float(np.nanmean(shot_rmse)),
        RMSE_std=float(np.nanstd(shot_rmse, ddof=0)),
        MAE=float(np.nanmean(shot_mae)),
        MAE_std=float(np.nanstd(shot_mae, ddof=0)),
        window_RMSE_mean=float(np.nanmean(rmse)),
        window_RMSE_median=float(np.nanmedian(rmse)),
        window_RMSE_p90=float(np.nanpercentile(rmse, 90)),
        window_RMSE_max=float(np.nanmax(rmse)),
    )


# ======================================================================================
# INFERENCE PASS
# ======================================================================================
def load_test_loader_and_model(split, batch=None):
    setup_imports()
    import torch
    loaders, meta = build_data(split, ["test"])
    if batch is not None:
        # rebuild the loader with the requested batch size (build_data uses CFG batch)
        from torch.utils.data import DataLoader
        from src.trainer import model_collate_fn
        from utils import seed_worker
        ds = loaders["test"].dataset
        nw = CFG["dl_workers"]
        if nw < 0:
            nw = max(2, min(8, os.cpu_count() or 2))
        g = torch.Generator(); g.manual_seed(CFG["seed"])
        loaders["test"] = DataLoader(ds, batch_size=batch, num_workers=nw, collate_fn=model_collate_fn,
                                     worker_init_fn=seed_worker, generator=g,
                                     pin_memory=torch.cuda.is_available(),
                                     prefetch_factor=2 if nw > 0 else None)
    model = make_model(loaders["test"], meta)
    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model.load_state_dict(torch.load(os.path.join(rob_paths(split)["model_dir"], "best_model.pt"),
                                     map_location=dev, weights_only=False))
    model.to(dev).eval()
    return loaders["test"], model, meta, dev


def train_channel_means(split):
    """Per-signal mean computed on the TRAIN subset (for mean-imputation, no leakage)."""
    setup_imports()
    try:
        P_ = P.paths(split)
        sel = json.load(open(P_["sel"]))
        train_shots = [s for s in sel.get("shots", {}).get("train", [])
                       if os.path.isdir(os.path.join(P_["data"], f"{s}.zarr"))]
        if len(train_shots) >= 5:
            loaders, _ = build_data(split, ["train"])
            return _train_means_pass(loaders["train"])
        else:
            log(f"only {len(train_shots)} train shots on disk; using baseline 0.0 for mean imputation")
            return {i: 0.0 for i in range(len(SIGNALS))}
    except Exception as e:  # noqa
        log("train-channel-mean pass failed, falling back to zeros:", repr(e)[:200])
        return {i: 0.0 for i in range(len(SIGNALS))}


def _train_means_pass(loader, max_batches=40):
    sums = None; cnts = None
    seen = 0
    for batch in loader:
        if batch is None:
            continue
        x = batch[2]
        if sums is None:
            sums = [0.0] * len(x); cnts = [0] * len(x)
        for i, t in enumerate(x):
            a = _as_np(t).astype(np.float64)
            a = a[np.isfinite(a) & (a != 0)]
            sums[i] += float(a.sum()); cnts[i] += int(a.size)
        seen += 1
        if seen >= max_batches:
            break
    means = {}
    for i in range(len(sums)):
        means[i] = (sums[i] / cnts[i]) if cnts[i] > 0 else 0.0
    log(f"train channel means (first {seen} batches): " +
        ", ".join(f"{SIGNAL_SHORT[i]}={means[i]:.3g}" for i in sorted(means)))
    return means


def run_pass(loader, model, dev, mu, sd, corrupt_fn=None, accumulate_spatial=False, return_viz=False):
    """Forward the whole test loader; return aggregated metrics + per-window rows.

    corrupt_fn: callable(x_list) -> (x_corrupted_list, info) applied per batch, or None
                for the clean pass.
    """
    import torch
    sid_all, wid_all, rmse_all, mae_all = [], [], [], []
    spatial_sum = None; spatial_cnt = None
    viz_data = None
    n_batches = 0
    with torch.no_grad():
        for batch in loader:
            if batch is None:
                continue
            shot_id, window_id, x, y = batch
            x = [t.to(dev, dtype=torch.float32) for t in x]
            xin = x
            if corrupt_fn is not None:
                xin_cpu, info = corrupt_fn(x)
                xin = [t.to(dev, dtype=torch.float32) for t in xin_cpu]
            y_pred = model(*xin)
            if not isinstance(y_pred, (list, tuple)):
                y_pred = [y_pred]
            yt = y[0].detach().cpu().squeeze(1).numpy().astype(np.float64) * sd + mu
            yp = y_pred[0].detach().cpu().squeeze(1).numpy().astype(np.float64) * sd + mu
            yt = yt.reshape(yt.shape[0], -1, 65, 65)
            yp = yp.reshape(yp.shape[0], -1, 65, 65)
            # flatten any extra window axis into the sample axis
            n = yt.shape[0] * yt.shape[1]
            yt2 = yt.reshape(n, 65, 65)
            yp2 = yp.reshape(n, 65, 65)
            if return_viz and viz_data is None and n > 0:
                sids_raw = np.asarray(shot_id).reshape(-1)
                viz_data = dict(yt=yt2[0].tolist(), yp=yp2[0].tolist(),
                                shot_id=int(sids_raw[0]) if len(sids_raw) > 0 else 0)
            rmse, mae = window_metrics(yt2, yp2)
            rmse = rmse.reshape(yt.shape[0], yt.shape[1]).mean(axis=1)
            mae = mae.reshape(yt.shape[0], yt.shape[1]).mean(axis=1)
            sid_all.append(np.asarray(shot_id).reshape(-1))
            wid_all.append(np.asarray(window_id).reshape(-1))
            rmse_all.append(rmse); mae_all.append(mae)
            if accumulate_spatial:
                err = np.abs(yp2 - yt2)
                s = np.nansum(err, axis=0); c = np.isfinite(err).sum(axis=0)
                spatial_sum = s if spatial_sum is None else spatial_sum + s
                spatial_cnt = c if spatial_cnt is None else spatial_cnt + c
            n_batches += 1
    sid = np.concatenate(sid_all) if len(sid_all) > 0 else np.array([])
    rmse = np.concatenate(rmse_all) if len(rmse_all) > 0 else np.array([])
    mae = np.concatenate(mae_all) if len(mae_all) > 0 else np.array([])
    m = aggregate(sid, rmse, mae, sd)
    m["n_batches"] = n_batches
    if accumulate_spatial and spatial_sum is not None and spatial_cnt is not None:
        m["spatial_abs_err"] = (spatial_sum / np.maximum(spatial_cnt, 1)).tolist()
    if return_viz and viz_data is not None:
        m["viz"] = viz_data
    m["_windows"] = dict(shot_id=sid.tolist(), rmse=rmse.tolist(), mae=mae.tolist())
    return m


def build_sweep_tasks(rc):
    """Expand the sweep into a list of task dicts (id, callable-corruption spec)."""
    tasks = []
    for scen in rc["scenarios"]:
        for sev in SEVERITIES[scen]:
            for imp in rc["imputes"]:
                if scen in ("category_abl", "group_failure"):
                    groups = CATEGORY_GROUPS if scen == "category_abl" else CORRELATED_GROUPS
                    for gname, gchans in groups.items():
                        tasks.append(dict(id=f"{scen}|{gname}|{imp}", scenario=scen,
                                          severity=sev, impute=imp, group=gname, chans=gchans))
                else:
                    tasks.append(dict(id=f"{scen}|{sev}|{imp}", scenario=scen,
                                      severity=sev, impute=imp, group=None, chans=None))
    return tasks


def make_corrupt_fn(task, rng_seed, train_means):
    """Return a callable(x_list)->(x_corrupt,info) with a fresh RNG per task (seeded)."""
    rng = np.random.default_rng(rng_seed)
    if task["scenario"] in ("category_abl", "group_failure"):
        def fn(x):
            return apply_group(x, task["scenario"], task["severity"], task["impute"],
                               train_means, rng, task["group"], task["chans"])
    else:
        def fn(x):
            return apply_scenario(x, task["scenario"], task["severity"], task["impute"],
                                  train_means, rng)
    return fn


# ======================================================================================
# COMMANDS
# ======================================================================================
def cmd_corrupt(split):
    """Build + persist the sweep manifest (and verify the clean baseline gate)."""
    rc = rob_cfg()
    P_ = rob_paths(split); os.makedirs(P_["dir"], exist_ok=True)
    tasks = build_sweep_tasks(rc)
    json.dump(dict(config=rc, tasks=tasks, signals=SIGNALS, categories=CATEGORY_GROUPS,
                   correlated_groups=CORRELATED_GROUPS, severities=SEVERITIES),
              open(P_["manifest"], "w"), indent=1)
    log(f"corruption manifest: {len(tasks)} tasks -> {P_['manifest']}")
    log("sweep tasks:\n  " + "\n  ".join(t["id"] for t in tasks))


def cmd_eval_robust(split):
    rc = rob_cfg()
    P_ = rob_paths(split); os.makedirs(P_["dir"], exist_ok=True)
    loader, model, meta, dev = load_test_loader_and_model(split, batch=rc["batch"])
    mu, sd = _psi_mean_std(meta)

    # ---- CLEAN GATE: must reproduce the eval-time NRMSE before corruption is trusted
    log("CLEAN pass (sensor-healthy baseline)...")
    clean = run_pass(loader, model, dev, mu, sd, accumulate_spatial=True, return_viz=True)
    viz_samples = dict(clean=clean.pop("viz", None), tasks={})
    log(f"CLEAN: NRMSE={clean['NRMSE']:.4f} (±{clean['NRMSE_std']:.4f}) NMAE={clean['NMAE']:.4f} "
        f"RMSE={clean['RMSE']:.5f} MAE={clean['MAE']:.5f} n_shots={clean['n_shots']} n_windows={clean['n_windows']}")
    targets = None
    mp = os.path.join(P_["model_dir"], "..", "results", "task_1-3", "task_metrics.csv")
    if os.path.exists(mp):
        import csv as _csv
        with open(mp) as fh:
            for row in _csv.DictReader(fh):
                feat = row.get("feature_name", "")
                if feat in ("equilibrium-psi", "equilibrium_psi", "task_1-3") or "psi" in feat:
                    val = row.get("NRMSE_mean") or row.get("NRMSE")
                    if val:
                        try:
                            targets = float(val)
                        except Exception:
                            pass
                        break
    if targets is not None:
        d = abs(clean["NRMSE"] - targets)
        ok = d <= max(0.02, 0.15 * targets)
        log(f"clean-gate: harness NRMSE={clean['NRMSE']:.4f} vs eval CSV={targets:.4f} "
            f"(Δ={d:.4f}) -> {'PASS' if ok else 'FAIL (trust nothing downstream)'}")
        if not ok:
            log("WARNING: clean baseline mismatch; corruption numbers below are suspect")

    result = dict(clean=clean, corrupted={}, config=rc, psi_mean=mu, psi_std=sd, viz_samples=viz_samples)
    if rc["clean_only"]:
        clean.pop("_windows", None)
        json.dump(result, open(P_["metrics"], "w"), indent=1)
        log("TM_ROB_CLEAN_ONLY=1 -> wrote clean-only metrics, skipping corruption")
        return

    log("building TRAIN channel means for mean-imputation...")
    train_means = train_channel_means(split)

    VIZ_TASKS = {"random_dropout|0.5|zero", "random_dropout|0.5|ffill",
                 "category_abl|plasma_current|zero", "category_abl|plasma_current|ffill"}
    tasks = build_sweep_tasks(rc)
    log(f"corruption sweep: {len(tasks)} tasks on the TEST split...")
    rows = []
    for ti, task in enumerate(tasks):
        cf = make_corrupt_fn(task, rc["seed"] * 1000003 + ti, train_means)
        need_viz = task["id"] in VIZ_TASKS
        log(f"[{ti + 1}/{len(tasks)}] {task['id']} ...")
        try:
            m = run_pass(loader, model, dev, mu, sd, corrupt_fn=cf, return_viz=need_viz)
        except Exception as e:  # noqa
            log(f"  task {task['id']} FAILED: {repr(e)[:200]}")
            continue
        if need_viz and "viz" in m:
            viz_samples["tasks"][task["id"]] = m.pop("viz", None)
        m.pop("_windows", None)
        m["id"] = task["id"]
        rs = clean["NRMSE"] / m["NRMSE"] if m["NRMSE"] > 0 else float("nan")
        m["RS_vs_clean"] = float(rs)
        result["corrupted"][task["id"]] = m
        log(f"  {task['id']}: NRMSE={m['NRMSE']:.4f} (+{100 * (m['NRMSE'] / clean['NRMSE'] - 1):.1f}%) "
            f"NMAE={m['NMAE']:.4f} RS={rs:.3f}")
        rows.append(dict(id=task["id"], scenario=task["scenario"], severity=task["severity"],
                         impute=task["impute"], group=task["group"], **{k: m[k] for k in (
                             "NRMSE", "NMAE", "RMSE", "MAE", "RS_vs_clean", "n_shots", "n_windows")}))
        json.dump(result, open(P_["metrics"], "w"), indent=1)  # incremental save

    clean.pop("_windows", None)
    json.dump(result, open(P_["metrics"], "w"), indent=1)
    log(f"robustness metrics written -> {P_['metrics']} ({len(rows)} corrupted tasks)")


def cmd_report(split):
    rc = rob_cfg()
    P_ = rob_paths(split)
    if not os.path.exists(P_["metrics"]):
        raise SystemExit(f"FATAL: {P_['metrics']} not found - run eval-robust first")
    res = json.load(open(P_["metrics"]))
    os.makedirs(P_["figures"], exist_ok=True)
    clean = res["clean"]; corrupted = res["corrupted"]

    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    # ---- RS aggregation: worst-case NRMSE per scenario (zero-fill), Gupta-style
    def rs_table():
        worst = {}
        for tid, m in corrupted.items():
            scen, _, imp = tid.split("|")
            if imp != "zero":
                continue
            worst[scen] = max(worst.get(scen, 0.0), m["NRMSE"])
        rs = {scen: (clean["NRMSE"] / v if v else float("nan")) for scen, v in worst.items()}
        return worst, rs

    worst, rs = rs_table()

    # ---- figure 1: degradation curves NRMSE vs severity, per scenario (zero-fill)
    try:
        fig, ax = plt.subplots(figsize=(8, 5))
        for scen in SCENARIOS:
            pts = []
            for tid, m in corrupted.items():
                parts = tid.split("|")
                if parts[0] == scen and parts[-1] == "zero" and len(parts) == 3:
                    pts.append((float(parts[1]), m["NRMSE"]))
            if pts:
                pts.sort()
                ax.plot([p[0] for p in pts], [p[1] for p in pts], marker="o", label=scen)
        ax.axhline(clean["NRMSE"], ls="--", c="k", lw=1, label="clean")
        ax.set_xlabel("severity"); ax.set_ylabel("NRMSE (test, un-standardised)")
        ax.set_title(f"Robustness degradation - {TASK} ({split})")
        ax.legend(fontsize=8); ax.grid(alpha=0.3); fig.tight_layout()
        fig.savefig(os.path.join(P_["figures"], "degradation_curves.png"), dpi=120)
        plt.close(fig)
        log("saved degradation_curves.png")
    except Exception as e:  # noqa
        log("degradation curve failed:", repr(e)[:200])

    # ---- figure 2: per-signal importance (category_abl + channel_ablation n=1), zero-fill
    try:
        labels, vals = [], []
        for tid, m in corrupted.items():
            p = tid.split("|")
            if p[0] == "category_abl" and p[-1] == "zero":
                labels.append(p[1]); vals.append(100 * (m["NRMSE"] / clean["NRMSE"] - 1))
        order = np.argsort(vals)[::-1]
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar([labels[i] for i in order], [vals[i] for i in order], color="steelblue")
        ax.set_ylabel("% NRMSE increase vs clean"); ax.grid(alpha=0.3, axis="y")
        ax.set_title(f"Diagnostic importance (zero each category) - {TASK}")
        fig.tight_layout(); fig.savefig(os.path.join(P_["figures"], "signal_importance.png"), dpi=120)
        plt.close(fig)
        log("saved signal_importance.png")
    except Exception as e:  # noqa
        log("importance bar failed:", repr(e)[:200])

    # ---- figure 3: spatial error map (clean)
    try:
        sm = np.array(clean.get("spatial_abs_err", []))
        if sm.size:
            fig, ax = plt.subplots(figsize=(4.5, 4))
            im = ax.imshow(sm, cmap="magma")
            ax.set_title("Clean |error| map (65x65)")
            ax.axis("off"); fig.colorbar(im, ax=ax, fraction=0.046)
            fig.tight_layout(); fig.savefig(os.path.join(P_["figures"], "spatial_error_clean.png"), dpi=120)
            plt.close(fig)
            log("saved spatial_error_clean.png")
    except Exception as e:  # noqa
        log("spatial map failed:", repr(e)[:200])

    # ---- figure 4: qualitative 2D psi under corruption & mitigation
    try:
        vs = res.get("viz_samples")
        if vs and vs.get("clean") and vs.get("tasks"):
            yt = np.array(vs["clean"]["yt"])
            yp_clean = np.array(vs["clean"]["yp"])
            sid = vs["clean"].get("shot_id", 0)

            t_drop_z = vs["tasks"].get("random_dropout|0.5|zero") or {}
            t_drop_f = vs["tasks"].get("random_dropout|0.5|ffill") or {}
            t_ip_z = vs["tasks"].get("category_abl|plasma_current|zero") or {}
            t_ip_f = vs["tasks"].get("category_abl|plasma_current|ffill") or {}

            yp_drop_z = np.array(t_drop_z.get("yp", yp_clean))
            yp_drop_f = np.array(t_drop_f.get("yp", yp_clean))
            yp_ip_z = np.array(t_ip_z.get("yp", yp_clean))
            yp_ip_f = np.array(t_ip_f.get("yp", yp_clean))

            fig, axes = plt.subplots(2, 4, figsize=(13, 6.2), layout="constrained")
            rows = [
                ("50% Random Sensor Dropout", yp_drop_z, yp_drop_f),
                ("Complete Plasma Current (Ip) Loss", yp_ip_z, yp_ip_f),
            ]
            for r_i, (label, yp_z, yp_f) in enumerate(rows):
                err_clean = float(np.mean(np.abs(yp_clean - yt)))
                err_z = float(np.mean(np.abs(yp_z - yt)))
                err_f = float(np.mean(np.abs(yp_f - yt)))
                cols = [
                    (yt, f"Ground Truth $\\psi$ (Shot {sid})", "viridis"),
                    (yp_clean, f"Clean Pred (MAE: {err_clean:.4f} Wb)", "viridis"),
                    (yp_z, f"Fault: {label} [Zero-Fill]\n(MAE: {err_z:.4f} Wb)", "inferno"),
                    (yp_f, f"Mitigated: [Forward-Fill]\n(MAE: {err_f:.4f} Wb)", "viridis"),
                ]
                for c_i, (img, ttl, cmap) in enumerate(cols):
                    ax = axes[r_i, c_i]
                    im = ax.imshow(img, cmap=cmap)
                    ax.set_title(ttl, fontsize=9.5, fontweight="semibold")
                    ax.axis("off")
                    fig.colorbar(im, ax=ax, fraction=0.046)
            out_psi = os.path.join(P_["figures"], "robustness_psi_reconstruction.png")
            fig.savefig(out_psi, dpi=150)
            plt.close(fig)
            log("saved robustness_psi_reconstruction.png")
    except Exception as e:  # noqa
        log("robustness psi plot failed:", repr(e)[:200])

    # ---- figure 5: diagnostic criticality ranking
    try:
        cats, vals = [], []
        for tid, m in corrupted.items():
            p = tid.split("|")
            if p[0] == "category_abl" and p[-1] == "zero":
                cats.append(p[1])
                vals.append(100 * (m["NRMSE"] / clean["NRMSE"] - 1))
        if cats:
            order = np.argsort(vals)
            max_v = max(vals) if vals else 100.0
            fig, ax = plt.subplots(figsize=(7.5, 3.8), layout="constrained")
            bars = ax.barh([cats[i] for i in order], [vals[i] for i in order], color="#2b5c8f", height=0.6)
            ax.axvline(0, color="black", linewidth=0.8)
            ax.set_xlim(0, max_v * 1.25)
            ax.set_xlabel("NRMSE Error Surge Above Clean Baseline (%)", fontsize=10, fontweight="bold")
            ax.set_title("Diagnostic Criticality: Test NRMSE Surge When Sensor Groups Fail", fontsize=11, fontweight="bold")
            for bar, idx in zip(bars, order):
                ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                        f"+{vals[idx]:.1f}%", va="center", ha="left", fontsize=9, fontweight="semibold")
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            fig.savefig(os.path.join(P_["figures"], "robustness_criticality.png"), dpi=200)
            plt.close(fig)
            log("saved robustness_criticality.png")
    except Exception as e:  # noqa
        log("robustness criticality failed:", repr(e)[:200])

    # ---- figure 6: online fault mitigation comparison
    try:
        drop_sevs = [0.10, 0.25, 0.50]
        z_vals = [clean["NRMSE"]]
        m_vals = [clean["NRMSE"]]
        f_vals = [clean["NRMSE"]]
        labels = ["Clean (p=0)"] + [f"p = {s:.2f}" for s in drop_sevs]
        for s in drop_sevs:
            z_vals.append(corrupted.get(f"random_dropout|{s}|zero", {}).get("NRMSE", clean["NRMSE"]))
            m_vals.append(corrupted.get(f"random_dropout|{s}|mean", {}).get("NRMSE", clean["NRMSE"]))
            f_vals.append(corrupted.get(f"random_dropout|{s}|ffill", {}).get("NRMSE", clean["NRMSE"]))
        x = np.arange(len(labels))
        width = 0.24
        fig, ax = plt.subplots(figsize=(7.5, 3.8), layout="constrained")
        ax.bar(x - width, z_vals, width, label="Zero-Fill (Default Baseline)", color="#dc2626")
        ax.bar(x, m_vals, width, label="Mean-Fill (Dataset Mean)", color="#f59e0b")
        ax.bar(x + width, f_vals, width, label="Forward-Fill (Physics-Informed)", color="#10b981")
        ax.axhline(clean["NRMSE"], color="#64748b", linestyle="--", linewidth=1.0, label="Clean Baseline")
        ax.set_ylabel("Test NRMSE (Lower is Better)", fontsize=10, fontweight="bold")
        ax.set_title("Online Fault Mitigation: Imputation Under Random Dropouts", fontsize=11, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=9.5, fontweight="semibold")
        ax.legend(frameon=True, fontsize=8.5, loc="upper left")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        fig.savefig(os.path.join(P_["figures"], "robustness_mitigation.png"), dpi=200)
        plt.close(fig)
        log("saved robustness_mitigation.png")
    except Exception as e:  # noqa
        log("robustness mitigation failed:", repr(e)[:200])

    # ---- write robustness.md
    lines = [f"# Robustness report - {TASK} ({split} split)", ""]
    lines += [
        "Sensor-failure study mirroring Gupta (arXiv 2607.11915), on the equilibrium",
        "reconstruction task (Gupta used task_4-4). Model: frozen `best_model.pt`.",
        "**Test split only. Model never retrained on corrupted data.**",
        "",
        "## Clean baseline (sensor-healthy)",
        f"- NRMSE = **{clean['NRMSE']:.4f}** (±{clean['NRMSE_std']:.4f})",
        f"- NMAE = {clean['NMAE']:.4f}, RMSE = {clean['RMSE']:.5f}, MAE = {clean['MAE']:.5f}",
        f"- n_shots = {clean['n_shots']}, n_windows = {clean['n_windows']}",
        "",
        "## Robustness Score (RS) — worst case per scenario, zero-fill",
        "RS = NRMSE_clean / NRMSE_corrupted; 1.0 = no damage, lower = worse.",
        "",
        "| scenario | worst NRMSE | RS |",
        "|---|---|---|",
    ]
    for scen in SCENARIOS:
        if scen in worst:
            lines.append(f"| {scen} | {worst[scen]:.4f} | {rs[scen]:.3f} |")
    lines += ["", "## Full sweep", "",
              "| task | NRMSE | NMAE | RS | n_shots |", "|---|---|---|---|---|"]
    for tid in sorted(corrupted):
        m = corrupted[tid]
        lines.append(f"| {tid} | {m['NRMSE']:.4f} | {m['NMAE']:.4f} | {m['RS_vs_clean']:.3f} | {m['n_shots']} |")
    lines += ["", "## Caveats", "",
              "- Subset data: absolute NRMSE is NOT the official full-set figure.",
              "- mean-imputation uses TRAIN-split channel means (no test leakage).",
              "- Corruption is inference-time only; a model retrained on damaged data",
              "  could behave differently.",
              ""]
    open(P_["report"], "w", encoding="utf-8").write("\n".join(lines))
    log(f"wrote {P_['report']}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["corrupt", "eval-robust", "report"])
    ap.add_argument("--split", default="random", choices=["random", "temporal"])
    args = ap.parse_args()
    log(f"robustness {args.cmd} split={args.split} task={TASK}")
    if args.cmd == "corrupt":
        cmd_corrupt(args.split)
    elif args.cmd == "eval-robust":
        cmd_eval_robust(args.split)
    else:
        cmd_report(args.split)
    log("done")


if __name__ == "__main__":
    main()


## 1. Data Preparation (Downloads the exact test shots)
Uses existing `out/random/selection.json` and `out/tar_index.json` to download the identical test data without re-indexing.


In [ ]:
run(f"cd /kaggle/working && TM_PREP_SPLITS=test python -u pipeline.py prep --split {DATA_SPLIT}")


## 2. Clean Gate Sanity Check
Runs inference on the clean test data to confirm the model reproduces the baseline NRMSE (~0.1419) before any corruption is tested.


In [ ]:
run(f"cd /kaggle/working && TM_ROB_CLEAN_ONLY=1 python -u robustness.py eval-robust --split {DATA_SPLIT}")


## 3. Generate Corruption Sweep Manifest
Builds the 63 corruption tasks across the 6 failure scenarios and 3 imputation modes.


In [ ]:
run(f"cd /kaggle/working && python -u robustness.py corrupt --split {DATA_SPLIT}")


## 4. Run the Full Robustness Sweep (~15-20 min)
Iterates through all 63 tasks on the test split. Metrics are saved incrementally to `out/random/robustness/robust_metrics.json`.


In [ ]:
run(f"cd /kaggle/working && python -u robustness.py eval-robust --split {DATA_SPLIT}")


## 5. Generate Report & Degradation Curves
Computes Robustness Scores (RS), degradation curves, signal importance bars, and spatial error maps.


In [ ]:
run(f"cd /kaggle/working && python -u robustness.py report --split {DATA_SPLIT}")


## 6. View Results Inline & Package Artifacts
Prints `robustness.md`, renders the figures, and creates `robustness_results.zip` in `/kaggle/working` for one-click download.


In [ ]:
from IPython.display import Image, display, Markdown

rdir = f"/kaggle/working/out/{DATA_SPLIT}/robustness"
rep_file = os.path.join(rdir, "robustness.md")

if os.path.exists(rep_file):
    print("=" * 60)
    print("ROBUSTNESS STUDY REPORT")
    print("=" * 60)
    display(Markdown(open(rep_file, encoding="utf-8").read()))

fig_dir = os.path.join(rdir, "figures")
if os.path.isdir(fig_dir):
    all_figs = [
        "robustness_psi_reconstruction.png",
        "robustness_criticality.png",
        "robustness_mitigation.png",
        "degradation_curves.png",
        "signal_importance.png",
        "spatial_error_clean.png",
    ]
    for f in all_figs:
        p = os.path.join(fig_dir, f)
        if os.path.exists(p):
            print(f"\nFigure: {f}")
            display(Image(filename=p))

# Zip existing results safely for easy download from Kaggle output section
cands = [f"out/{DATA_SPLIT}/robustness", f"out/{DATA_SPLIT}/figures", f"out/{DATA_SPLIT}/results", f"out/{DATA_SPLIT}/selection.json", f"out/{DATA_SPLIT}/model"]
targets = [p for p in cands if os.path.exists(f"/kaggle/working/{p}")]
run(f"cd /kaggle/working && zip -q -r robustness_results.zip {' '.join(targets)}")
print("\n✓ All robustness artifacts packaged to /kaggle/working/robustness_results.zip")
